# Notebook 05 — Mendelian Genetics and Punnett Squares

**Module:** 05 — Biology Fundamentals  
**Notebook:** 05 of 18  
**Prerequisites:** NB04  
**Time estimate:** 60 minutes

---
## Step 1 — Motivation

Carrier frequency calculations, disease inheritance risk prediction, GWAS interpretation —
all depend on the logic Mendel derived in 1865. When you see 'autosomal recessive' in a
clinical genetics paper or OMIM entry, this notebook is what allows you to work out the
probability that a child is affected given their parents' genotypes.

---
## Step 2 — Intuition

Each parent contributes **one** allele to each offspring — chosen at random from their
two copies. A Punnett square is nothing more than an exhaustive probability table of
which allele each parent contributes. Two parents, each with two alleles → 2 × 2 = 4
equally likely allele combinations.

---
## Step 3 — Biological Background

**Mendel's laws (1865):**
1. **Law of Segregation:** alleles separate during gamete formation; each gamete
   carries exactly one allele per locus.
2. **Law of Independent Assortment:** alleles at different loci assort independently
   during gamete formation *(only true for unlinked genes; genes on the same chromosome
   violate this — important for GWAS interpretation)*.

**Terminology:**
- **Genotype:** the alleles an organism carries (AA, Aa, aa)
- **Phenotype:** the observable trait produced by the genotype
- **Dominant allele (A):** expressed even when one copy is present (Aa shows A phenotype)
- **Recessive allele (a):** only expressed when two copies are present (aa)
- **Homozygous:** both alleles are the same (AA or aa)
- **Heterozygous:** alleles differ (Aa) — often called a *carrier* for recessive diseases

**Inheritance modes:**

| Mode | Example disease | Carrier risk |
|------|----------------|-------------|
| Autosomal dominant | Huntington's, BRCA1/2 | 50% children affected if one parent affected |
| Autosomal recessive | Cystic fibrosis, PKU | 25% affected if both parents carriers |
| X-linked recessive | Hemophilia A, colour blindness | Males affected; females carriers |
| Codominant | ABO blood type | Both alleles expressed |

**Why Mendel matters for GWAS:**
GWAS tests whether alleles at a locus are associated with disease. The expected
allele frequencies under random mating (Hardy-Weinberg) form the null hypothesis.
Understanding Mendelian ratios tells you what to expect in the absence of selection.

---
## Step 4 — Mathematical Explanation

**Monohybrid cross (Aa × Aa):**

| | A | a |
|--|--|--|
| **A** | AA | Aa |
| **a** | Aa | aa |

Genotype ratio: 1 AA : 2 Aa : 1 aa → phenotype ratio 3 dominant : 1 recessive

**Dihybrid cross (AaBb × AaBb):**  
4 × 4 = 16 combinations → 9 A_B_ : 3 A_bb : 3 aaB_ : 1 aabb

**Cystic fibrosis carrier risk example:**
- CFTR gene: F508del is a common recessive allele  
- Two carrier parents (Aa × Aa):  
  P(affected child) = 1/4; P(carrier child) = 1/2; P(unaffected homozygous) = 1/4
- If the general population carrier frequency is q, by Hardy-Weinberg the probability
  that two unrelated individuals are both carriers = q²

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from itertools import product
from collections import Counter, defaultdict

In [ ]:
# Cell 6.1 — Punnett square generator (monohybrid and dihybrid)
def punnett_square(parent1: str, parent2: str) -> dict:
    """
    Generate a Punnett square for one locus.
    Inputs: parent1, parent2 as genotype strings, e.g. 'Aa', 'AA', 'aa'
    Returns: dict with 'grid' (2x2 list), 'genotype_counts', 'genotype_freqs'
    """
    assert len(parent1) == 2 and len(parent2) == 2, "Each parent must have exactly 2 alleles"
    alleles1 = list(parent1)   # each gamete carries one allele
    alleles2 = list(parent2)

    grid = [[a1 + a2 for a2 in alleles2] for a1 in alleles1]

    # Normalise genotype: always put uppercase first (AA before Aa, Aa before aa)
    def normalise(geno):
        return ''.join(sorted(geno, key=lambda c: (c.islower(), c)))

    offspring = [normalise(grid[i][j]) for i in range(2) for j in range(2)]
    counts = Counter(offspring)
    total = sum(counts.values())
    freqs = {g: c / total for g, c in counts.items()}

    return {'grid': grid, 'genotype_counts': counts, 'genotype_freqs': freqs}

# Example 1: carrier × carrier (autosomal recessive disease)
result = punnett_square('Aa', 'Aa')
print("Cross: Aa × Aa (carrier × carrier)")
print("Punnett grid:")
for row in result['grid']:
    print("  ", row)
print("Genotype frequencies:")
for g, f in sorted(result['genotype_freqs'].items()):
    print(f"  {g}: {f:.2f}  ({f:.0%})")

In [ ]:
# Cell 6.2 — Autosomal dominant: one parent affected (Aa × aa)
print("\nCross: Aa × aa (autosomal dominant, one parent affected)")
r2 = punnett_square('Aa', 'aa')
for g, f in sorted(r2['genotype_freqs'].items()):
    print(f"  {g}: {f:.2f}")
print(f"  Probability child is affected: {sum(f for g,f in r2['genotype_freqs'].items() if 'A' in g.upper() and g != g.lower()):.0%}")

# X-linked recessive: carrier mother × unaffected father
# Mother: X^A X^a (carrier), Father: X^A Y
# Sons: X^A or X^a Y — 50% of sons affected
print("\nX-linked recessive: carrier mother × unaffected father")
print("  Daughters: 50% carriers (X^A X^a), 50% unaffected (X^A X^A)")
print("  Sons:      50% affected (X^a Y),    50% unaffected (X^A Y)")

In [ ]:
# Cell 6.3 — Simulate 10,000 offspring and verify Mendelian ratios
rng = np.random.default_rng(42)

def simulate_offspring(parent1, parent2, n=10000):
    alleles1 = list(parent1)
    alleles2 = list(parent2)

    def normalise(geno):
        return ''.join(sorted(geno, key=lambda c: (c.islower(), c)))

    gametes1 = rng.choice(alleles1, size=n)
    gametes2 = rng.choice(alleles2, size=n)
    offspring = [normalise(g1 + g2) for g1, g2 in zip(gametes1, gametes2)]
    return Counter(offspring)

sim = simulate_offspring('Aa', 'Aa', n=10_000)
expected = {'AA': 0.25, 'Aa': 0.50, 'aa': 0.25}

print("Carrier × Carrier cross (n=10,000 simulated offspring):")
print(f"  {'Genotype':<10} {'Expected':>10} {'Observed':>10}")
for g in ['AA', 'Aa', 'aa']:
    obs = sim[g] / 10000
    print(f"  {g:<10} {expected[g]:>10.3f} {obs:>10.3f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Visual Punnett square + inheritance mode comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Punnett square visualization for Aa × Aa
ax = axes[0]
ax.set_xlim(0, 3); ax.set_ylim(0, 3); ax.axis('off')
ax.set_title('Punnett Square: Aa × Aa (carrier × carrier)', fontsize=11)

# Headers
ax.text(1.5, 2.75, 'Parent 2', ha='center', fontweight='bold', fontsize=10)
ax.text(2.5, 2.5, 'A', ha='center', va='center', fontsize=12, color='steelblue')
ax.text(2.5+1, 2.5, 'a', ha='center', va='center', fontsize=12, color='steelblue')

ax.text(0.25, 1.5, 'Parent 1', va='center', fontweight='bold', fontsize=10, rotation=90)
ax.text(0.75, 1.5+0.5, 'A', ha='center', va='center', fontsize=12, color='tomato')
ax.text(0.75, 1.5-0.5, 'a', ha='center', va='center', fontsize=12, color='tomato')

cells_info = [
    (2.5, 2.0, 'AA', '#d4edda', 'Unaffected homozygous'),
    (3.5, 2.0, 'Aa', '#fff3cd', 'Carrier'),
    (2.5, 1.0, 'Aa', '#fff3cd', 'Carrier'),
    (3.5, 1.0, 'aa', '#f8d7da', 'Affected'),
]
for x, y, label, color, tooltip in cells_info:
    rect = patches.FancyBboxPatch((x-0.45, y-0.45), 0.9, 0.9,
                                   boxstyle='round,pad=0.05', facecolor=color,
                                   edgecolor='gray', lw=1)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(1, 4.5); ax.set_ylim(0.3, 3.1)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#d4edda', label='AA: unaffected (25%)'),
                   Patch(facecolor='#fff3cd', label='Aa: carrier (50%)'),
                   Patch(facecolor='#f8d7da', label='aa: affected (25%)')]
ax.legend(handles=legend_elements, loc='lower center', fontsize=9)

# Panel 2: Genotype proportions across inheritance modes
ax = axes[1]
modes = ['Aa × Aa\n(rec. carrier)', 'Aa × aa\n(dom. affected)', 'AA × aa\n(dominant)', 'aa × aa\n(rec. affected)']
crosses = [('Aa','Aa'), ('Aa','aa'), ('AA','aa'), ('aa','aa')]
colors_geno = {'AA': '#d4edda', 'Aa': '#fff3cd', 'aa': '#f8d7da'}

x = np.arange(len(modes))
width = 0.25
for i, geno in enumerate(['AA', 'Aa', 'aa']):
    freqs = []
    for p1, p2 in crosses:
        r = punnett_square(p1, p2)
        freqs.append(r['genotype_freqs'].get(geno, 0))
    ax.bar(x + (i-1)*width, freqs, width, label=geno,
           color=list(colors_geno.values())[i], edgecolor='gray', lw=0.8)

ax.set_xticks(x); ax.set_xticklabels(modes, fontsize=8)
ax.set_ylabel('Offspring genotype frequency')
ax.set_title('Mendelian genotype ratios\nacross 4 cross types')
ax.legend(title='Genotype')
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `punnett_square(parent1, parent2)` from scratch. Test it on:
   - `'AA'` × `'Aa'` (expected: 50% AA, 50% Aa)
   - `'Aa'` × `'aa'` (expected: 50% Aa, 50% aa)
2. Cystic fibrosis (CFTR gene) is autosomal recessive. The carrier frequency in
   Europeans is approximately 1/25. What is the probability that two random
   Europeans both carry the CFTR mutation? If they do both carry it, what is
   the probability their child is affected?
3. In X-linked recessive inheritance, why are males almost always the ones affected
   (not females)? What would need to happen for a female to be affected?
4. Look up ABO blood type inheritance. ABO is codominant. Cross a type AB person
   (I^A I^B) with a type O person (ii). What blood types can their children have?
   Build the Punnett square by hand.

---
## Quiz — Active Recall

1. What are Mendel's two laws? State each in one sentence.
2. What is the difference between genotype and phenotype?
3. Two carrier parents (Aa × Aa): what fraction of children will be affected?
   What fraction will be carriers?
4. Why does autosomal dominant disease not skip generations?
5. Why does X-linked recessive disease appear much more often in males than females?

---
## Reflection

**Date completed:** ____________________

> *[A patient has cystic fibrosis (autosomal recessive). Their sibling wants to
> know their carrier probability before having children. Both parents are unaffected.
> What is the probability the sibling is a carrier? Work through the logic step by step.]*

---
*Next: `06_population_genetics_hardy_weinberg.ipynb`*